## Local validation metric comparisons

These plots compare the computed simulation metrics in two paired ways: policy B minus policy A, and bystander minus arrival. The script treats `report.json`/`manifest.json` as the authoritative run condition and uses the latest matching CSV row for low-level metric scores.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

notebook_dir = Path.cwd()
repo_root = notebook_dir.parents[1] if notebook_dir.name == "analysis" else notebook_dir
tools_dir = repo_root / "tools"
if str(tools_dir) not in sys.path:
    sys.path.insert(0, str(tools_dir))

from plot_local_validation_metrics import build_visualizations

local_validation_root = repo_root / "artifacts/local-validation"
metric_viz_dir = repo_root / "artifacts/local-validation/metric-visualizations"

metric_viz_dir = build_visualizations(local_validation_root, metric_viz_dir)
print(f"Updated metric dashboard: {metric_viz_dir / 'index.html'}")

summary = pd.read_csv(metric_viz_dir / "condition_summary.csv")
manifest_rows = []
for manifest_path in sorted(local_validation_root.glob("q*/policy*/manifest.json")):
    manifest = json.loads(manifest_path.read_text())
    manifest_rows.append(
        {
            "q_bucket": manifest_path.parents[1].name,
            "run_dir": manifest_path.parent.relative_to(local_validation_root).as_posix(),
            "trace_backend": manifest.get("trace_backend") or "unknown/legacy",
            "execution_backend_id": manifest.get("execution_backend_id") or "unknown",
            "trace_source": manifest.get("trace_source") or "unknown",
            "real_backend_verified": manifest.get("real_backend_verified", False),
        }
    )

manifest_df = pd.DataFrame(manifest_rows)
provenance = summary.merge(manifest_df, on=["q_bucket", "run_dir"], how="left")
provenance_columns = [
    "q_bucket",
    "folder_name",
    "episode",
    "policy",
    "perspective",
    "terminal_status",
    "trace_backend",
    "execution_backend_id",
    "trace_source",
    "real_backend_verified",
    "selected_csv",
]
display(provenance[[column for column in provenance_columns if column in provenance.columns]])

reference_runs = provenance[provenance["trace_backend"].fillna("unknown/legacy") != "real"]
if len(reference_runs):
    print("Runs not confirmed as --trace-backend real:")
    display(
        reference_runs[
            [
                "q_bucket",
                "folder_name",
                "episode",
                "policy",
                "perspective",
                "trace_backend",
                "execution_backend_id",
            ]
        ]
    )
else:
    print("All included runs are real backend traces.")

warnings_path = metric_viz_dir / "data_warnings.txt"
warnings = warnings_path.read_text().strip() if warnings_path.exists() else ""
print(warnings if warnings else "No data warnings.")


In [ ]:
from IPython.display import HTML, SVG, display

display(HTML(f'<p><a href="{metric_viz_dir / "index.html"}" target="_blank">Open the full metric dashboard</a></p>'))

for svg_name, title in [
    ("axis_score_heatmap.svg", "Axis scores by condition"),
    ("policy_delta_heatmap.svg", "Policy deltas: policy B minus policy A"),
    ("perspective_delta_heatmap.svg", "Perspective deltas: bystander minus arrival"),
    ("feature_metric_heatmap.svg", "Normalized feature metric scores"),
]:
    display(HTML(f"<h3>{title}</h3>"))
    display(SVG(filename=str(metric_viz_dir / svg_name)))


In [ ]:
import pandas as pd

policy_deltas = pd.read_csv(metric_viz_dir / "policy_deltas.csv")
perspective_deltas = pd.read_csv(metric_viz_dir / "perspective_deltas.csv")

display(policy_deltas.reindex(policy_deltas["delta_policy_b_minus_a"].abs().sort_values(ascending=False).index).head(8))
display(perspective_deltas.reindex(perspective_deltas["delta_bystander_minus_arrival"].abs().sort_values(ascending=False).index).head(8))


### Local validation metric histograms

These histograms use the same simple count-based reading as the human perception plots above. The scores are normalized from 0 to 1, so the bins show how many computed metric values land in each quality band.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

if "metric_viz_dir" not in globals():
    metric_viz_dir = Path("artifacts/local-validation/metric-visualizations")

condition_summary = pd.read_csv(metric_viz_dir / "condition_summary.csv")

axis_columns = [
    "axis_perceived_dexterity",
    "axis_perceived_safety",
    "axis_perceived_social_awareness",
    "axis_impression",
]

axis_labels = {
    "axis_perceived_dexterity": "dexterity",
    "axis_perceived_safety": "safety",
    "axis_perceived_social_awareness": "social awareness",
    "axis_impression": "impression",
}

score_bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.000001]
score_bin_labels = ["0.0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]


def count_scores_by_bin(values):
    numeric_values = pd.to_numeric(values, errors="coerce").dropna().clip(0.0, 1.0)
    binned = pd.cut(
        numeric_values,
        bins=score_bins,
        labels=score_bin_labels,
        include_lowest=True,
        right=False,
    )
    return binned.value_counts().reindex(score_bin_labels, fill_value=0)


def plot_score_histogram(long_table, group_column, score_column, title):
    groups = list(long_table[group_column].dropna().drop_duplicates())
    x_positions = list(range(len(score_bin_labels)))
    bar_width = 0.8 / max(len(groups), 1)

    plt.figure(figsize=(9, 4.5))
    for group_index, group in enumerate(groups):
        group_values = long_table.loc[long_table[group_column] == group, score_column]
        counts = count_scores_by_bin(group_values)
        offsets = [x - 0.4 + (bar_width / 2) + group_index * bar_width for x in x_positions]
        plt.bar(offsets, counts.values, width=bar_width, label=f"{group} (n={len(group_values.dropna())})")

    plt.xticks(x_positions, score_bin_labels)
    plt.xlabel("Normalized metric score")
    plt.ylabel("Number of computed scores")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


axis_scores_long = condition_summary.melt(
    id_vars=["episode", "policy", "perspective", "terminal_status"],
    value_vars=axis_columns,
    var_name="axis",
    value_name="score",
).dropna(subset=["score"])
axis_scores_long["axis"] = axis_scores_long["axis"].map(axis_labels)

plot_score_histogram(
    axis_scores_long,
    group_column="policy",
    score_column="score",
    title="Computed axis score distribution by policy",
)

plot_score_histogram(
    axis_scores_long,
    group_column="perspective",
    score_column="score",
    title="Computed axis score distribution by perspective",
)

metric_columns = [column for column in condition_summary.columns if column.startswith("metric_")]
metric_scores_long = condition_summary.melt(
    id_vars=["episode", "policy", "perspective", "terminal_status"],
    value_vars=metric_columns,
    var_name="metric",
    value_name="score",
).dropna(subset=["score"])

plot_score_histogram(
    metric_scores_long,
    group_column="policy",
    score_column="score",
    title="Computed feature metric score distribution by policy",
)
